In [157]:
import os
os.environ["HTTP_PROXY"]= "http://proxy.utwente.nl:3128"
os.environ["HTTPS_PROXY"]= "http://proxy.utwente.nl:3128"
os.environ["http_proxy"]= "http://proxy.utwente.nl:3128"
os.environ["https_proxy"]= "http://proxy.utwente.nl:3128"

Hierarchy only

In [161]:
import json
from collections import defaultdict

# ============================================================
# CONFIG
# ============================================================
BOSCH_FILE   = "datasets/tram_train.json"
CLEANED_FILE = "../../data_augmentatio_stefano/mitre/mitre_relationships.json"
OUTPUT_FILE  = "datasets/tram_augmented_hierarchy.json"
# ============================================================
# 1) Load BOSCH (to get allowed labels)
# ============================================================
with open(BOSCH_FILE, "r", encoding="utf-8") as f:
    bosch = json.load(f)

# Collect all unique labels currently used in your training set
bosch_labels = bosch["labels"]
allowed_labels = set(l for labs in bosch_labels.values() for l in labs)

print(f"BOSCH unique labels: {len(allowed_labels)}")

# ============================================================
# 2) Load augmentation records
# ============================================================
with open(CLEANED_FILE, "r", encoding="utf-8") as f:
    cleaned = json.load(f)

print(f"Cleaned augmentation records: {len(cleaned)}")

# ============================================================
# 3) Group augmentation data by Parent Family
# ============================================================
# We need to see the "family" structure to apply the rules
parent_to_subs = defaultdict(set)   # Parent -> {Sub1, Sub2, ...}
parent_to_sents = defaultdict(list) # Parent -> [(tid, sentence), ...]

def get_parent(tid: str) -> str:
    return tid.split(".")[0] if tid and "." in tid else tid

for item in cleaned:
    tid = item.get("technique_id")
    sent = (item.get("relationship_description") or "").strip()
    if not tid or not sent:
        continue

    parent = get_parent(tid)
    if "." in tid:
        parent_to_subs[parent].add(tid)
    
    parent_to_sents[parent].append((tid, sent))

# ============================================================
# 4) Apply Policy Rules
# ============================================================
new_sentences = {}
new_labels = {}
next_idx = 0

for parent, family_sents in parent_to_sents.items():
    # Identify which parts of this family are in our allowed list
    subs_in_family = parent_to_subs[parent]
    allowed_subs = [s for s in subs_in_family if s in allowed_labels]
    parent_is_allowed = parent in allowed_labels

    # --- RULE 1: Parent allowed, NO sub-techniques allowed ---
    if parent_is_allowed and len(allowed_subs) == 0:
        for tid, sent in family_sents:
            # Policy: add sentences of sub-techniques, label as parent
            if "." in tid:
                idx = str(next_idx)
                new_sentences[idx] = sent
                new_labels[idx] = [parent]
                next_idx += 1

# --- RULE 2: Sole sub-technique allowed (no siblings, no parent) ---
    elif not parent_is_allowed and len(allowed_subs) == 1:
        target_sub = allowed_subs[0]
        for tid, sent in family_sents:
            # Removed 'if tid != target_sub' to ensure "ALL" sentences 
            # (including the sub-technique's own sentences) are in the final file.
            idx = str(next_idx)
            new_sentences[idx] = sent
            new_labels[idx] = [target_sub]
            next_idx += 1

# ============================================================
# 5) Save Output
# ============================================================
doc_titles = [
    f"mitre_hierarchy_augmented_{i}"
    for i in range(len(new_sentences))
]

output = {
    "sentence": new_sentences,
    "labels": new_labels,
    "doc_title": {k: f"mitre_hierarchy_augmented_{i}" 
              for i, k in enumerate(new_sentences.keys())}
}

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(output, f, indent=2, ensure_ascii=False)

print(f"Augmentation complete.")
print(f"Total new sentences added: {len(new_sentences)}")

BOSCH unique labels: 50
Cleaned augmentation records: 19280
Augmentation complete.
Total new sentences added: 7958


Hierarchy + embeddings

In [167]:
import json
from collections import defaultdict, Counter
import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# ============================================================
# CONFIG
# ============================================================
BOSCH_FILE   = "datasets/bosch_train.json"
CLEANED_FILE = "../../data_augmentatio_stefano/mitre/mitre_relationships.json"
OUTPUT_FILE  = "datasets/bosch_augmented_hierarchy_embeddings.json"

SIM_THRESHOLD = 0.98
MAX_SEMANTIC_PER_LABEL = 10000

# ============================================================
# 1) Load BOSCH (to get allowed labels)
# ============================================================
with open(BOSCH_FILE, "r", encoding="utf-8") as f:
    bosch = json.load(f)

bosch_sentences = bosch["sentence"]
bosch_labels = bosch["labels"]
allowed_labels = set(l for labs in bosch_labels.values() for l in labs)

print(f"BOSCH unique labels: {len(allowed_labels)}")

# ============================================================
# 2) Load augmentation records
# ============================================================
with open(CLEANED_FILE, "r", encoding="utf-8") as f:
    cleaned = json.load(f)

print(f"Cleaned augmentation records: {len(cleaned)}")

# ============================================================
# 3) Group augmentation data by Parent Family
# ============================================================
parent_to_subs = defaultdict(set)
parent_to_sents = defaultdict(list)

def get_parent(tid: str) -> str:
    return tid.split(".")[0] if tid and "." in tid else tid

for item in cleaned:
    tid = item.get("technique_id")
    sent = (item.get("relationship_description") or "").strip()
    if not tid or not sent:
        continue

    parent = get_parent(tid)
    if "." in tid:
        parent_to_subs[parent].add(tid)

    parent_to_sents[parent].append((tid, sent))

# ============================================================
# 4) Apply Policy Rules (HIERARCHY ONLY — UNCHANGED)
# ============================================================
new_sentences = {}
new_labels = {}
next_idx = 0

for parent, family_sents in parent_to_sents.items():
    subs_in_family = parent_to_subs[parent]
    allowed_subs = [s for s in subs_in_family if s in allowed_labels]
    parent_is_allowed = parent in allowed_labels

    # --- RULE 1: Parent allowed, NO sub-techniques allowed ---
    if parent_is_allowed and len(allowed_subs) == 0:
        for tid, sent in family_sents:
            if "." in tid:
                idx = str(next_idx)
                new_sentences[idx] = sent
                new_labels[idx] = [parent]
                next_idx += 1

    # --- RULE 2: Sole sub-technique allowed ---
    elif not parent_is_allowed and len(allowed_subs) == 1:
        target_sub = allowed_subs[0]
        for tid, sent in family_sents:
            idx = str(next_idx)
            new_sentences[idx] = sent
            new_labels[idx] = [target_sub]
            next_idx += 1

print(f"[Hierarchy] Added samples: {len(new_sentences)}")

# ============================================================
# 5) Prepare hierarchy projection function (for semantic stage)
# ============================================================
def hierarchical_project(tid: str):
    if tid in allowed_labels:
        return tid

    parent = get_parent(tid)
    if parent in allowed_labels:
        return parent

    subs = parent_to_subs.get(parent, set())
    allowed_subs = [s for s in subs if s in allowed_labels]

    if len(allowed_subs) == 1:
        return allowed_subs[0]

    return None

# ============================================================
# 6) Build label descriptions (for semantic fallback)
# ============================================================
bosch_label_desc = {}
cleaned_label_desc = {}

for item in cleaned:
    tid = item.get("technique_id")
    name = item.get("technique_name")
    desc = item.get("technique_description") or item.get("description")

    if not tid or not name:
        continue

    full_text = name if not desc else f"{name}. {desc}"
    cleaned_label_desc[tid] = full_text

    if tid in allowed_labels:
        bosch_label_desc[tid] = full_text

assert bosch_label_desc, "No BOSCH label descriptions found for semantic matching"

# ============================================================
# 7) Load SecureBERT
# ============================================================
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading SecureBERT on {device}")

secbert = SentenceTransformer("ehsanaghaei/SecureBERT").to(device)

def embed(text: str):
    return secbert.encode(
        [text],
        device=device,
        convert_to_numpy=True,
        normalize_embeddings=True
    )[0]

# ============================================================
# 8) Embed BOSCH labels once
# ============================================================
bosch_labels_sorted = sorted(bosch_label_desc.keys())
bosch_embeddings = np.vstack([
    embed(bosch_label_desc[l]) for l in bosch_labels_sorted
])

# ============================================================
# 9) Semantic fallback projection
# ============================================================
def semantic_project(tid: str):
    desc = cleaned_label_desc.get(tid)
    if not desc:
        return None

    q = embed(desc).reshape(1, -1)
    sims = cosine_similarity(q, bosch_embeddings)[0]

    best_idx = int(np.argmax(sims))
    best_score = float(sims[best_idx])

    if best_score < SIM_THRESHOLD:
        return None

    return bosch_labels_sorted[best_idx]

# ============================================================
# 10) Add semantic-only augmentations (ON TOP of hierarchy)
# ============================================================
semantic_added = 0
per_label_cap = Counter()

for item in cleaned:
    sent = (item.get("relationship_description") or "").strip()
    tid = item.get("technique_id")

    if not sent or not tid:
        continue

    # hierarchy has priority
    if hierarchical_project(tid) is not None:
        continue

    mapped = semantic_project(tid)
    if not mapped:
        continue

    if per_label_cap[mapped] >= MAX_SEMANTIC_PER_LABEL:
        continue

    idx = str(next_idx)
    new_sentences[idx] = sent
    new_labels[idx] = [mapped]
    next_idx += 1

    per_label_cap[mapped] += 1
    semantic_added += 1

print(f"[Semantic] Added samples: {semantic_added}")

# ============================================================
# 11) Save Output
# ============================================================
doc_titles = [
    f"mitre_hierarchy_augmented_{i}"
    for i in range(len(new_sentences))
]

output = {
    "sentence": new_sentences,
    "labels": new_labels,
    "doc_title": {k: f"mitre_hierarchy_augmented_{i}" 
              for i, k in enumerate(new_sentences.keys())}
}

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(output, f, indent=2, ensure_ascii=False)

print("Augmentation complete.")
print(f"Total samples: {len(new_sentences)}")


BOSCH unique labels: 203
Cleaned augmentation records: 19280
[Hierarchy] Added samples: 9819
Loading SecureBERT on cuda


No sentence-transformers model found with name ehsanaghaei/SecureBERT. Creating a new one with mean pooling.
Some weights of RobertaModel were not initialized from the model checkpoint at ehsanaghaei/SecureBERT and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


[Semantic] Added samples: 157
Augmentation complete.
Total samples: 9976
